In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
# Create a copy of the dataset
df = dataset.copy()

# Convert categorical columns to numeric values
df['sex'] = df['sex'].map({'male': 0, 'female': 1})
df['smoker'] = df['smoker'].map({'yes': 1, 'no': 0})
df['region'] = df['region'].map({'southwest': 0, 'southeast': 1, 'northwest': 2, 'northeast': 3})

# Split data into training (80%) and testing (20%) sets
train_dataset = df.sample(frac=0.8, random_state=0)
test_dataset = df.drop(train_dataset.index)

# Separate the 'expenses' column (this is our target for prediction)
train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

print("Data processing complete!")

In [ ]:
# 1. Building an Improved Model
model = keras.Sequential([
    layers.Input(shape=(len(train_dataset.keys()),)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'), # Additional hidden layer
    layers.Dense(1)
])

# 2. Compile the model (Using Adam for better accuracy)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='mse',
    metrics=['mae', 'mse']
)

# 3. Model Training (Increasing epochs to 1000)
print("Training started... please wait.")
history = model.fit(
    train_dataset,
    train_labels,
    epochs=1000,
    validation_split=0.2,
    verbose=0
)
print("Training finished! Now run the next cell to check results.")

In [ ]:
# Train the model for 500 epochs
history = model.fit(
    train_dataset,
    train_labels,
    epochs=500,
    validation_split=0.2,
    verbose=0
)

print("Training finished!")

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
